# VORTEX-AI (CG-NSDE) - Kaggle GPU training

Headless training run for Contrastive Graph-Neural Stochastic Differential Equations.

**Mounts**

- `vortex-ai-code` - training source, extracted to `/kaggle/working/repo` because Kaggle inputs
  are read-only and Lightning must write checkpoints.
- `vortex-ai-data` - the preprocessed `.npy` artifacts, linked in at `repo/data/raw`. Nothing is
  downloaded or rebuilt, so every run trains on exactly the arrays used locally.

**Outputs** land in `/kaggle/working` and are pulled down with
`kaggle kernels output brxkeaaryan/vortex-ai-cgnsde-2026 -p <dir>`.

Set `SMOKE = True` in the run-config cell for a 2-epoch validation run before committing GPU
quota to the full 200 epochs.

## 1. Environment

In [ ]:
import glob
import importlib
import os
import shutil
import subprocess
import sys

print("python", sys.version.split()[0])

import torch

print("torch", torch.__version__, "| cuda", torch.version.cuda)
print("gpus", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(" ", i, torch.cuda.get_device_name(i))

## 2. Dependencies

Kaggle already ships a CUDA-matched PyTorch, so it is never reinstalled - only missing packages
are added. `torch-geometric` comes from plain PyPI because this repo uses `GATConv` and
`dense_to_sparse` only, not the `torch_scatter` / `torch_sparse` compiled extensions.

In [ ]:
REQUIRED = {
    "torch_geometric": "torch-geometric",
    "torchsde": "torchsde",
    "pytorch_lightning": "pytorch-lightning",
    "wandb": "wandb",
}

missing = []
for module, package in REQUIRED.items():
    try:
        importlib.import_module(module)
    except ImportError:
        missing.append(package)

if missing:
    print("installing:", missing, flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("all dependencies already present")

import pytorch_lightning
import torch_geometric
import torchsde

print("torch_geometric", torch_geometric.__version__)
print("torchsde", torchsde.__version__)
print("pytorch_lightning", pytorch_lightning.__version__)

## 3. Stage code and link the data

Kaggle unpacks an uploaded archive server-side, so the code normally arrives as
`/kaggle/input/vortex-ai-code/repo`. Both that and a raw `repo.tar` are handled.

In [ ]:
WORK = "/kaggle/working/repo"
if os.path.exists(WORK):
    shutil.rmtree(WORK)

print("=== /kaggle/input ===")
for root, dirnames, filenames in os.walk("/kaggle/input"):
    depth = root[len("/kaggle/input"):].count(os.sep)
    if depth <= 2:
        print("  " * depth + (os.path.basename(root) or "/") + "/")
        for name in filenames[:6]:
            print("  " * (depth + 1) + name)

tars = glob.glob("/kaggle/input/**/repo.tar", recursive=True)
dirs = sorted(
    {os.path.dirname(p) for p in glob.glob("/kaggle/input/**/train_vortex.py", recursive=True)},
    key=len,
)

if dirs:
    shutil.copytree(dirs[0], WORK)
elif tars:
    os.makedirs(WORK)
    subprocess.run(["tar", "-xf", tars[0], "-C", WORK], check=True)
    if not os.path.exists(os.path.join(WORK, "train_vortex.py")):
        nested = os.path.join(WORK, "repo")
        if os.path.exists(os.path.join(nested, "train_vortex.py")):
            WORK = nested
else:
    raise RuntimeError("code dataset not mounted under /kaggle/input")

os.chdir(WORK)
print("workdir:", os.getcwd())

# Point data/raw at the read-only artifact mount. A symlink avoids copying 374 MB per run.
data_mounts = sorted(
    {os.path.dirname(p) for p in glob.glob("/kaggle/input/**/windows.npy", recursive=True)},
    key=len,
)
assert data_mounts, "data dataset not mounted under /kaggle/input"
raw = os.path.join(WORK, "data", "raw")
if os.path.islink(raw):
    os.unlink(raw)
elif os.path.isdir(raw):
    shutil.rmtree(raw)
os.symlink(data_mounts[0], raw)
print("data/raw ->", os.path.realpath(raw))

In [ ]:
import numpy as np

windows = np.load("data/raw/windows.npy")
adj = np.load("data/raw/adj_matrices.npy")
feats = np.load("data/raw/window_node_features.npy")

print("windows     ", windows.shape)
print("adj_matrices", adj.shape)
print("node_feats  ", feats.shape)

n, T, N = windows.shape
assert (T, N) == (60, 50), f"unexpected window shape {windows.shape}"
assert adj.shape == (n, N, N), f"unexpected adjacency shape {adj.shape}"
# window_node_features comes from build_scaler_split and can be one window shorter than the
# window/adjacency arrays; train_vortex.py aligns all three with min() at load time.
assert feats.shape[1:] == (T, N, 6), f"unexpected node-feature shape {feats.shape}"
assert feats.shape[0] in (n, n - 1), f"node-feature length {feats.shape[0]} vs {n} windows"

for name in ("scaler.npz", "split.npz", "window_regimes_v2.npy", "volume_windows.npy", "nifty50_close.csv"):
    assert os.path.exists(f"data/raw/{name}"), f"missing artifact: {name}"

print(f"OK: {n} windows x {T} days x {N} stocks")

## 4. Train

`--devices 1` on purpose: the model is tiny (50 nodes, batch 4), so a second T4 would add
synchronisation cost rather than throughput.

In [ ]:
import re

EXPECTED_CODE = "v19-regime-only-balanced"
src = open("train_vortex.py", encoding="utf-8").read()
m = re.search(r'CODE_VERSION\s*=\s*"([^"]+)"', src)
assert m, "code dataset is stale: train_vortex.py has no CODE_VERSION marker"
print("mounted code version:", m.group(1))
assert m.group(1) == EXPECTED_CODE, (
    f"code dataset is stale: mounted {m.group(1)!r}, notebook expects {EXPECTED_CODE!r}. "
    "Re-stage with .\\kaggle\\run.ps1 code and wait for the dataset version to publish."
)
assert "_nll_per_asset" in open("training/vortex_model.py", encoding="utf-8").read(), "stale training code"
print("code freshness OK")


In [ ]:
SMOKE = False          # True -> 2-epoch validation run
MAX_EPOCHS = 2 if SMOKE else 200
BATCH_SIZE = 16
NUM_WORKERS = 2
RUN_TAG = "kaggle_smoke" if SMOKE else "kaggle_t4"

cmd = [
    "python", "train_vortex.py",
    "--config", "training/config.yaml",
    "--accelerator", "cuda",
    "--devices", "1",
    "--max_epochs", str(MAX_EPOCHS),
    "--batch_size", str(BATCH_SIZE),
    "--num_workers", str(NUM_WORKERS),
    "--run_tag", RUN_TAG,
]

print("$ " + " ".join(cmd), flush=True)
subprocess.run(cmd, check=True, cwd=WORK)

## 5. Evaluate the best checkpoint

Skipped in smoke mode - a 2-epoch model says nothing about scenario quality.

In [ ]:
best_file = "models/checkpoints/best_model_path.txt"
if os.path.exists(best_file):
    best = open(best_file).read().strip()
    assert os.path.exists(best), f"recorded best checkpoint missing: {best}"
else:
    checkpoints = glob.glob("models/checkpoints/*.ckpt")
    assert checkpoints, "no checkpoints were written"
    best = max(checkpoints, key=os.path.getmtime)
print("checkpoint:", best)
print("priors:", os.path.exists("models/checkpoints/generation_priors.npz"))
os.makedirs("results", exist_ok=True)
subprocess.run(
    [
        "python", "evaluate_vortex.py",
        "--checkpoint", best,
        "--device", "cuda",
        "--batch_size", str(BATCH_SIZE),
        "--output", "results/eval_kaggle.json",
    ],
    check=True,
    cwd=WORK,
)


## 6. Outputs

Everything under `/kaggle/working` is packaged with this kernel version.

In [ ]:
for root, _, files in os.walk(WORK):
    for f in files:
        if f.endswith((".ckpt", ".json", ".csv", ".log")):
            path = os.path.join(root, f)
            print(f"{os.path.getsize(path) / 1e6:9.2f} MB  {os.path.relpath(path, WORK)}")